# Projet X-Ray Pneumonia

## Objectif
D?tecter automatiquement la pneumonie sur des radiographies thoraciques avec une version am?lior?e du notebook d'origine.

## Ce qui change pour am?liorer le score global
- suppression du GAN, qui ajoutait des images synth?tiques 64x64 puis r?agrandies
- passage ? un mod?le de **transfer learning DenseNet121**, plus robuste pour l'imagerie m?dicale
- **split train / validation par groupe patient** pour limiter les fuites entre patients
- gestion du d?s?quilibre avec **class weights** au lieu d'un faux r??quilibrage partiel
- suivi de l'entra?nement avec **val_auc** plut?t que **val_accuracy**
- **optimisation du seuil** sur la validation au lieu d'un seuil fig? ? `0.45`

## Architecture du notebook
1. Imports & configuration
2. T?l?chargement du dataset
3. Fonctions utilitaires & audit
4. Analyse exploratoire rapide
5. Split train / validation
6. Pipeline TensorFlow
7. DenseNet121 + fine-tuning
8. ?valuation finale & m?triques compl?tes
9. Sauvegarde du mod?le final
10. Pr?diction sur une nouvelle image


In [ ]:
# ==========================
# 1. IMPORTS & CONFIGURATION
# ==========================
import gc
import json
import math
import os
import random
import warnings
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None
    from sklearn.model_selection import GroupShuffleSplit
else:
    GroupShuffleSplit = None

from tensorflow import keras
from tensorflow.keras import callbacks, layers, regularizers

warnings.filterwarnings("ignore")
plt.style.use("default")

# Reproductibilit?
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Constantes globales
IMG_SIZE = 224
BATCH_SIZE = 16
INITIAL_EPOCHS = 6
FINE_TUNE_EPOCHS = 8
FINE_TUNE_LAYERS = 60
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5
DROPOUT_RATE = 0.35
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
CLASS_NAMES = {0: "Pneumonia", 1: "Normal"}
VALID_EXTENSIONS = {".jpeg", ".jpg", ".png"}

print("TensorFlow version :", tf.__version__)
print("GPUs detectes      :", gpus)
print("Seed               :", SEED)
print("IMG_SIZE           :", IMG_SIZE)


In [ ]:
# ==========================
# 2. TELECHARGEMENT DU DATASET
# ==========================
dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
dataset_root = Path(dataset_path) / "chest_xray"

print("dataset_root :", dataset_root)
for subset_name in ["train", "val", "test"]:
    subset_path = dataset_root / subset_name
    print(f"{subset_name:>5} exists :", subset_path.exists(), "->", subset_path)


In [ ]:
# ==========================
# 3. FONCTIONS UTILITAIRES
# ==========================
def is_valid(path):
    return Path(path).suffix.lower() in VALID_EXTENSIONS


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and is_valid(p)])


def read_gray(path, size=None):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Impossible de lire : {path}")
    if size is not None:
        img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    return img


def show_grid(files, title, n=4):
    files = files[:n]
    if not files:
        print(f"Aucune image pour {title}")
        return

    cols = 2
    rows = math.ceil(len(files) / cols)
    plt.figure(figsize=(10, 3 * rows))
    for i, file_path in enumerate(files, 1):
        plt.subplot(rows, cols, i)
        plt.imshow(read_gray(file_path), cmap="gray")
        plt.title(f"{title} #{i}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


def extract_patient_group(path):
    stem = Path(path).stem
    if stem.startswith("person"):
        return stem.split("_")[0]
    if stem.startswith("IM-"):
        parts = stem.split("-")
        return "-".join(parts[:2]) if len(parts) >= 2 else stem
    return stem.split("_")[0].split("-")[0]


def build_subset_frame(subset_root, subset_name):
    rows = []
    subset_root = Path(subset_root)
    for label_value, label_name in CLASS_NAMES.items():
        class_dir = subset_root / label_name.upper()
        if not class_dir.exists():
            continue
        for file_path in list_images(class_dir):
            try:
                with Image.open(file_path) as img:
                    img.verify()
                rows.append({
                    "subset": subset_name,
                    "path": str(file_path),
                    "label": label_value,
                    "label_name": label_name,
                    "group": extract_patient_group(file_path),
                })
            except Exception:
                pass
    return pd.DataFrame(rows)


In [ ]:
# ==========================
# 4. AUDIT + ANALYSE EXPLORATOIRE
# ==========================
train_frames = []
for subset_name in ["train", "val"]:
    subset_path = dataset_root / subset_name
    if subset_path.exists():
        train_frames.append(build_subset_frame(subset_path, subset_name))

train_pool = pd.concat(train_frames, ignore_index=True)
test_df = build_subset_frame(dataset_root / "test", "test")
all_df = pd.concat([train_pool, test_df], ignore_index=True)

summary = (
    all_df.groupby(["subset", "label_name"]).size()
    .reset_index(name="count")
    .sort_values(["subset", "label_name"])
)
display(summary)

print("\nTotal images train+val :", len(train_pool))
print("Total images test      :", len(test_df))
print("Total groups train+val :", train_pool["group"].nunique())
print("Total groups test      :", test_df["group"].nunique())

normal_files = list_images(dataset_root / "train" / "NORMAL")
pneumonia_files = list_images(dataset_root / "train" / "PNEUMONIA")

show_grid(normal_files, "NORMAL", n=4)
show_grid(pneumonia_files, "PNEUMONIA", n=4)


In [ ]:
# ==========================
# 5. SPLIT REEL TRAIN / VAL
# ==========================
if StratifiedGroupKFold is not None:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, val_idx = next(
        splitter.split(
            X=train_pool["path"],
            y=train_pool["label"],
            groups=train_pool["group"],
        )
    )
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(
        splitter.split(
            X=train_pool["path"],
            y=train_pool["label"],
            groups=train_pool["group"],
        )
    )

train_df = train_pool.iloc[train_idx].reset_index(drop=True)
val_df = train_pool.iloc[val_idx].reset_index(drop=True)

def print_split_info(name, df):
    counts = df["label_name"].value_counts().sort_index()
    print(f"\n{name}")
    print("-" * len(name))
    print(counts.to_string())
    print("Total images :", len(df))
    print("Total groups :", df["group"].nunique())

print_split_info("Train", train_df)
print_split_info("Validation", val_df)
print_split_info("Test", test_df)


In [ ]:
# ==========================
# 6. PIPELINE TENSORFLOW
# ==========================
def decode_and_resize(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=1, expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)


def dataframe_to_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].astype("float32").values))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = dataframe_to_dataset(train_df, training=True)
val_ds = dataframe_to_dataset(val_df, training=False)
test_ds = dataframe_to_dataset(test_df, training=False)

print("train batches :", tf.data.experimental.cardinality(train_ds).numpy())
print("val batches   :", tf.data.experimental.cardinality(val_ds).numpy())
print("test batches  :", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
# ==========================
# 7. DATA AUGMENTATION + MODEL
# ==========================
tf.keras.backend.clear_session()
gc.collect()

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="xray_image")
x = data_augmentation(inputs)
x = layers.Concatenate(axis=-1, name="to_rgb")([x, x, x])
x = layers.Normalization(
    axis=-1,
    mean=[0.485, 0.456, 0.406],
    variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2],
    name="imagenet_preprocess",
)(x)

try:
    base_model = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    print("DenseNet121 charge avec poids ImageNet.")
except Exception as exc:
    print("Impossible de charger les poids ImageNet, fallback poids aleatoires :", exc)
    base_model = keras.applications.DenseNet121(
        include_top=False,
        weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )

base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
x = layers.Dropout(DROPOUT_RATE, name="dropout_head_1")(x)
x = layers.Dense(
    256,
    activation="relu",
    kernel_regularizer=regularizers.l2(WEIGHT_DECAY),
    name="dense_head",
)(x)
x = layers.Dropout(0.20, name="dropout_head_2")(x)
outputs = layers.Dense(1, activation="sigmoid", name="probability_normal")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="pneumonia_densenet121")

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=INITIAL_LR, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

model.summary()


In [ ]:
# ==========================
# 8. CLASS WEIGHTS + CALLBACKS
# ==========================
class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
class_weight_map = {0: float(class_weights_values[0]), 1: float(class_weights_values[1])}
print("class_weight_map :", class_weight_map)

ARTIFACT_DIR = Path.cwd() / "pneumonia_final_outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = ARTIFACT_DIR / "best_pneumonia_model.keras"

def build_callbacks():
    return [
        callbacks.ModelCheckpoint(
            filepath=BEST_MODEL_PATH,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            patience=2,
            factor=0.3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


In [ ]:
# ==========================
# 9. ENTRAINEMENT - PHASE 1
# ==========================
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weight_map,
    callbacks=build_callbacks(),
    verbose=1,
)


In [ ]:
# ==========================
# 10. FINE-TUNING
# ==========================
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-FINE_TUNE_LAYERS:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.AUC(name="auc"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=INITIAL_EPOCHS,
    epochs=INITIAL_EPOCHS + FINE_TUNE_EPOCHS,
    class_weight=class_weight_map,
    callbacks=build_callbacks(),
    verbose=1,
)

model = keras.models.load_model(BEST_MODEL_PATH, safe_mode=False)
print("Meilleur modele recharge depuis :", BEST_MODEL_PATH)


In [ ]:
# ==========================
# 11. COURBES D'APPRENTISSAGE
# ==========================
history_dict = {}
for history in [history_head, history_ft]:
    for key, values in history.history.items():
        history_dict.setdefault(key, []).extend(values)

history_df = pd.DataFrame(history_dict)
display(history_df.tail())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history_df["loss"], label="Train")
axes[0].plot(history_df["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history_df["accuracy"], label="Train")
axes[1].plot(history_df["val_accuracy"], label="Val")
axes[1].set_title("Accuracy")
axes[1].legend()

axes[2].plot(history_df["auc"], label="Train")
axes[2].plot(history_df["val_auc"], label="Val")
axes[2].set_title("AUC")
axes[2].legend()

plt.tight_layout()
plt.show()

history_df.to_csv(ARTIFACT_DIR / "history.csv", index=False)


In [ ]:
# ==========================
# 12. OPTIMISATION DU SEUIL SUR LA VALIDATION
# ==========================
y_val = val_df["label"].to_numpy(dtype="int32")
y_val_prob = model.predict(val_ds, verbose=1).ravel()

threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.01):
    y_val_pred = (y_val_prob >= threshold).astype("int32")
    threshold_rows.append({
        "threshold": float(np.round(threshold, 2)),
        "macro_f1": float(
            __import__("sklearn.metrics", fromlist=["f1_score"]).f1_score(
                y_val, y_val_pred, average="macro", zero_division=0
            )
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, y_val_pred)),
        "accuracy": float(np.mean(y_val == y_val_pred)),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df = threshold_df.sort_values(
    ["macro_f1", "balanced_accuracy", "accuracy", "threshold"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

BEST_THRESHOLD = float(threshold_df.loc[0, "threshold"])
display(threshold_df.head(10))
print("BEST_THRESHOLD :", BEST_THRESHOLD)

threshold_df.to_csv(ARTIFACT_DIR / "threshold_search.csv", index=False)


In [ ]:
# ==========================
# 13. EVALUATION FINALE SUR LE TEST
# ==========================
y_test = test_df["label"].to_numpy(dtype="int32")
y_test_prob = model.predict(test_ds, verbose=1).ravel()
y_test_pred = (y_test_prob >= BEST_THRESHOLD).astype("int32")

test_loss = model.evaluate(test_ds, verbose=0)[0]
test_acc = float(np.mean(y_test_pred == y_test))
test_auc = float(roc_auc_score(y_test, y_test_prob))
test_bal_acc = float(balanced_accuracy_score(y_test, y_test_pred))

print(f"Loss              : {test_loss:.4f}")
print(f"Accuracy          : {test_acc * 100:.2f} %")
print(f"Balanced Accuracy : {test_bal_acc:.4f}")
print(f"AUC               : {test_auc:.4f}")
print(f"Seuil optimal     : {BEST_THRESHOLD:.2f}")
print()

report_text = classification_report(
    y_test,
    y_test_pred,
    target_names=["Pneumonia (0)", "Normal (1)"],
    zero_division=0,
)
print(report_text)

report_dict = classification_report(
    y_test,
    y_test_pred,
    target_names=["Pneumonia (0)", "Normal (1)"],
    zero_division=0,
    output_dict=True,
)


In [ ]:
# ==========================
# 14. MATRICE DE CONFUSION
# ==========================
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(
    cm,
    display_labels=["Pneumonia (0)", "Normal (1)"],
).plot(values_format="d", cmap="Blues")
plt.title("Matrice de confusion - DenseNet121 fine-tuned")
plt.tight_layout()
plt.show()


In [ ]:
# ==========================
# 15. SAUVEGARDE DU MODELE FINAL
# ==========================
final_model_path = Path.cwd() / "pneumonia_cnn_model.keras"
final_metadata_path = Path.cwd() / "pneumonia_cnn_model.metadata.json"

model.save(final_model_path)

metadata = {
    "image_size": IMG_SIZE,
    "recommended_threshold": BEST_THRESHOLD,
    "class_mapping": {"0": "Pneumonia", "1": "Normal"},
    "model_name": model.name,
    "test_accuracy": test_acc,
    "test_auc": test_auc,
}
final_metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

evaluation_payload = {
    "best_threshold": BEST_THRESHOLD,
    "test_loss": float(test_loss),
    "test_accuracy": test_acc,
    "test_balanced_accuracy": test_bal_acc,
    "test_auc": test_auc,
    "classification_report": report_dict,
    "confusion_matrix": cm.tolist(),
}
(ARTIFACT_DIR / "evaluation.json").write_text(json.dumps(evaluation_payload, indent=2), encoding="utf-8")

backend_dir = Path.cwd() / "backend"
if backend_dir.exists():
    backend_model_path = backend_dir / "pneumonia_cnn_model.keras"
    backend_metadata_path = backend_dir / "pneumonia_cnn_model.metadata.json"
    model.save(backend_model_path)
    backend_metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print("Modele exporte aussi vers :", backend_model_path)

print("modele sauvegarde ici   :", final_model_path.resolve())
print("metadata sauvegardee ici:", final_metadata_path.resolve())
print("artifacts dir           :", ARTIFACT_DIR.resolve())


In [ ]:
# ==========================
# 16. CHARGEMENT DU MODELE SAUVEGARDE
# ==========================
loaded_model = keras.models.load_model(final_model_path, safe_mode=False)
print("modele recharge avec succes")
loaded_model.summary()


In [ ]:
# ==========================
# 17. PREDICTION SUR UNE NOUVELLE IMAGE
# ==========================
def preprocess_new_xray(image_path, img_size=IMG_SIZE):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Impossible de lire l'image : {image_path}")

    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    img = img.astype("float32") / 255.0
    img = np.expand_dims(img, axis=-1)
    img = np.expand_dims(img, axis=0)
    return img


def predict_new_xray(image_path, model, threshold=BEST_THRESHOLD):
    img = preprocess_new_xray(image_path)
    prob = model.predict(img, verbose=0).ravel()[0]
    pred = int(prob >= threshold)
    label = CLASS_NAMES[pred]

    print(f"probabilite classe 1 (Normal) : {prob:.4f}")
    print(f"seuil utilise                  : {threshold:.2f}")
    print(f"classe predite                 : {pred} -> {label}")

    return prob, pred


In [ ]:
# ==========================
# 18. EXEMPLE DE PREDICTION
# ==========================
sample_normal_path = Path(test_df[test_df["label"] == 1].iloc[0]["path"])
prob, pred = predict_new_xray(sample_normal_path, loaded_model, threshold=BEST_THRESHOLD)

img_show = cv2.imread(str(sample_normal_path), cv2.IMREAD_GRAYSCALE)
plt.figure(figsize=(4, 4))
plt.imshow(img_show, cmap="gray")
plt.title("Radiographie testee")
plt.axis("off")
plt.show()


## Notes finales

- Ce notebook garde le meme objectif que la version d'origine, mais remplace la partie GAN + CNN maison par une approche plus fiable pour maximiser le score global.
- La metrique a privilegier reste **AUC**, puis **macro F1** et **balanced accuracy** apres calibration du seuil.
- Si tu veux pousser encore plus loin, la prochaine etape logique serait un ensemble de plusieurs folds ou un second backbone comme EfficientNetV2B0 pour comparer.
